In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP02 - TP High Value
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   Percentage of "high value" business arrangements/contracts maintained by the unit 
   during the assessment period.
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the new Master AU List + TP Unit Mapping, strictly 
      filtered by 'Jurisdiction' for Canada, Hong Kong, and Barbados.
   2. TPRM STRING MAPPING: Maps TP engagements to AU IDs by checking if the 
      'TPRM_Units' string from the mapping table exists inside the text of the 
      'owning_lob' or 'lob_using' columns.
   3. HIGHEST VALUE FILTER: Cleans the 'Contract_Amount' string (removes commas/$) 
      and casts to a Double to find contracts >= 1,000,000.
   4. AGGREGATING (NUMERATOR & DENOMINATOR): Counts total DISTINCT engagements 
      (Denominator) and total DISTINCT engagements >= 1MM (Numerator).
   5. FINAL OUTPUT: Calculates the percentage (Numerator/Denominator * 100), rounds 
      to 2 decimal places, appends a '%' sign, and outputs the strict 6 columns.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data and strictly filter by target jurisdictions
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment 
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
),

Mapped_AUs AS (
    -- 2. Grab every AU that currently has TPRM units mapped to it
    SELECT DISTINCT TRIM(CAST(Assessable_Unit_ID AS STRING)) AS AU_ID 
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
),

All_Base_AUs AS (
    -- 3. Combine filtered Master List and Mapping to create the ultimate base table
    SELECT 
        COALESCE(mast.BusinessID, map.AU_ID) AS Base_AU_ID,
        mast.AU_Name,
        mast.Business_Segment,
        CASE WHEN mast.BusinessID IS NOT NULL THEN 'Yes' ELSE 'No' END AS In_ABAC_AU_List
    FROM Master_AUs mast
    FULL OUTER JOIN Mapped_AUs map 
        ON mast.BusinessID = map.AU_ID
),

Third_Party_Risk_Data AS (
    -- 4. Extract TP data, clean the contract amount string, and cast to double
    SELECT 
        EngagementNumber,
        owning_lob,
        lob_using,
        TRY_CAST(REPLACE(REPLACE(TRIM(Contract_Amount), ',', ''), '$', '') AS DOUBLE) AS Cleaned_Amount
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

Engagements_By_AU AS (
    -- 5. Map engagements using the TPRM unit mapping table and calculate Num/Denom
    SELECT 
        TRIM(CAST(map.Assessable_Unit_ID AS STRING)) AS Mapped_AU_ID,
        -- Total Distinct Engagements mapped to this AU
        COUNT(DISTINCT tp.EngagementNumber) AS Denominator,
        -- Distinct Engagements mapped to this AU that are >= 1 Million
        COUNT(DISTINCT CASE WHEN tp.Cleaned_Amount >= 1000000 THEN tp.EngagementNumber END) AS Numerator
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping map
    -- Joins if the mapped TPRM string is found inside owning_lob or lob_using
    INNER JOIN Third_Party_Risk_Data tp
        ON UPPER(tp.owning_lob) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
        OR UPPER(tp.lob_using) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
    WHERE map.Assessable_Unit_ID IS NOT NULL
    GROUP BY TRIM(CAST(map.Assessable_Unit_ID AS STRING))
)

-- 6. Final Template: Strict 6-column output with Percentage Math
SELECT 
    a.Base_AU_ID AS BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP02' AS QuestionID,               
    
    -- Math: Handles division by zero safely, calculates percentage, and appends '%'
    CASE 
        WHEN COALESCE(e.Denominator, 0) = 0 THEN '0%'
        ELSE CAST(ROUND((CAST(e.Numerator AS DOUBLE) / e.Denominator) * 100, 2) AS STRING) || '%'
    END AS Response,
    
    a.In_ABAC_AU_List
    
FROM All_Base_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.Base_AU_ID = e.Mapped_AU_ID
ORDER BY a.Base_AU_ID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP02 - TP High Value Drill-Down
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   PURPOSE: 
   Provides a row-by-row view of every Third Party Engagement mapped to an AU.
   Displays the mapping validation strings, the raw Contract Amount string, the 
   cleaned numeric value, and strictly flags if it counts toward the Numerator 
   (>= 1,000,000) or just the Denominator.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
),

Mapped_AUs AS (
    SELECT DISTINCT TRIM(CAST(Assessable_Unit_ID AS STRING)) AS AU_ID 
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
),

All_Base_AUs AS (
    SELECT 
        COALESCE(mast.BusinessID, map.AU_ID) AS Base_AU_ID,
        mast.AU_Name,
        CASE WHEN mast.BusinessID IS NOT NULL THEN 'Yes' ELSE 'No' END AS In_ABAC_AU_List
    FROM Master_AUs mast
    FULL OUTER JOIN Mapped_AUs map 
        ON mast.BusinessID = map.AU_ID
),

Third_Party_Risk_Data AS (
    SELECT 
        EngagementNumber,
        ThirdPartyName,
        owning_lob,
        lob_using,
        Contract_Amount AS Raw_Contract_Amount,
        Currency_code,
        TRY_CAST(REPLACE(REPLACE(TRIM(Contract_Amount), ',', ''), '$', '') AS DOUBLE) AS Cleaned_Amount
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
)

SELECT DISTINCT
    a.Base_AU_ID AS BusinessID,
    a.AU_Name,
    a.In_ABAC_AU_List,
    tp.EngagementNumber,
    tp.ThirdPartyName,
    map.TPRM_Units AS Successfully_Mapped_String,
    tp.owning_lob AS Raw_Col_K_owning_lob,
    tp.lob_using AS Raw_Col_L_lob_using,
    tp.Raw_Contract_Amount,
    tp.Currency_code,
    tp.Cleaned_Amount,
    
    CASE 
        WHEN tp.Cleaned_Amount >= 1000000 THEN 'Yes (Added to Numerator)'
        ELSE 'No (Denominator Only)' 
    END AS Is_High_Value_Contract
    
FROM All_Base_AUs a

-- Inner join AU list to mapping table
INNER JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map
    ON a.Base_AU_ID = TRIM(CAST(map.Assessable_Unit_ID AS STRING))

-- Inner join mapping string to raw TP data via LIKE
INNER JOIN Third_Party_Risk_Data tp
    ON UPPER(tp.owning_lob) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
    OR UPPER(tp.lob_using) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
    
ORDER BY BusinessID;